In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# TensorFlow and Keras imports
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam, RMSprop, SGD
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.wrappers.scikit_learn import KerasRegressor
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

# Visualization and utilities
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


ModuleNotFoundError: No module named 'tensorflow.keras.wrappers.scikit_learn'

In [ ]:
# Comprehensive RNN Evaluation and Hyperparameter Tuning

class RNNExperiment:
    """
    A comprehensive class for RNN experimentation and evaluation
    """
    
    def __init__(self):
        self.results = []
        self.best_model = None
        self.best_score = float('inf')
        
    def create_model(self, model_type='LSTM', units=50, layers=1, dropout=0.2, 
                    learning_rate=0.001, input_shape=(60, 1)):
        """
        Create RNN model with specified hyperparameters
        
        Args:
            model_type: 'RNN', 'LSTM', or 'GRU'
            units: Number of units in each layer
            layers: Number of recurrent layers
            dropout: Dropout rate
            learning_rate: Learning rate for optimizer
            input_shape: Shape of input data
            
        Returns:
            Compiled model
        """
        model = Sequential()
        
        # Add recurrent layers
        for i in range(layers):
            return_sequences = (i < layers - 1)  # Return sequences for all but last layer
            
            if i == 0:  # First layer needs input shape
                if model_type == 'LSTM':
                    model.add(LSTM(units, return_sequences=return_sequences, 
                                 input_shape=input_shape))
                elif model_type == 'GRU':
                    model.add(GRU(units, return_sequences=return_sequences, 
                                input_shape=input_shape))
                else:  # SimpleRNN
                    model.add(SimpleRNN(units, return_sequences=return_sequences, 
                                      input_shape=input_shape))
            else:
                if model_type == 'LSTM':
                    model.add(LSTM(units, return_sequences=return_sequences))
                elif model_type == 'GRU':
                    model.add(GRU(units, return_sequences=return_sequences))
                else:  # SimpleRNN
                    model.add(SimpleRNN(units, return_sequences=return_sequences))
            
            model.add(Dropout(dropout))
        
        # Output layer
        model.add(Dense(1, activation='linear'))
        
        # Compile model
        optimizer = Adam(learning_rate=learning_rate)
        model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])
        
        return model
    
    def evaluate_model(self, model, X_train, y_train, X_test, y_test, 
                      epochs=50, batch_size=32, verbose=0):
        """
        Train and evaluate a model
        
        Returns:
            Dictionary with evaluation results
        """
        # Callbacks
        early_stopping = EarlyStopping(monitor='val_loss', patience=10, 
                                     restore_best_weights=True)
        reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, 
                                    patience=5, min_lr=1e-7)
        
        # Train model
        history = model.fit(
            X_train, y_train,
            validation_split=0.2,
            epochs=epochs,
            batch_size=batch_size,
            callbacks=[early_stopping, reduce_lr],
            verbose=verbose
        )
        
        # Evaluate on test set
        test_loss = model.evaluate(X_test, y_test, verbose=0)
        predictions = model.predict(X_test, verbose=0)
        
        # Calculate additional metrics
        mse = mean_squared_error(y_test, predictions)
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, predictions)
        
        return {
            'test_loss': test_loss[0],
            'test_mae': test_loss[1],
            'mse': mse,
            'mae': mae,
            'rmse': rmse,
            'r2': r2,
            'history': history,
            'predictions': predictions
        }

# Initialize experiment class
experiment = RNNExperiment()
print("RNN Experiment class initialized successfully!")


In [ ]:
# Systematic Hyperparameter Tuning

def run_hyperparameter_search():
    """
    Run systematic hyperparameter search across different configurations
    """
    # Generate synthetic data for consistent comparison
    def create_synthetic_timeseries(n_samples=1000, seq_length=60):
        """Create synthetic time series data"""
        np.random.seed(42)
        t = np.linspace(0, 100, n_samples)
        
        # Combine multiple patterns
        trend = 0.01 * t
        seasonal = 2 * np.sin(2 * np.pi * t / 50) + 0.5 * np.sin(2 * np.pi * t / 10)
        noise = np.random.normal(0, 0.5, n_samples)
        
        data = trend + seasonal + noise
        
        # Create sequences
        X, y = [], []
        for i in range(seq_length, len(data)):
            X.append(data[i-seq_length:i])
            y.append(data[i])
        
        return np.array(X), np.array(y)
    
    # Generate data
    X, y = create_synthetic_timeseries()
    X = X.reshape((X.shape[0], X.shape[1], 1))
    
    # Split data
    train_size = int(0.7 * len(X))
    test_size = int(0.2 * len(X))
    
    X_train = X[:train_size]
    y_train = y[:train_size]
    X_test = X[train_size:train_size+test_size]
    y_test = y[train_size:train_size+test_size]
    
    # Normalize data
    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train.reshape(-1, 1)).flatten()
    y_test_scaled = scaler.transform(y_test.reshape(-1, 1)).flatten()
    
    # Define hyperparameter grid
    param_grid = {
        'model_type': ['SimpleRNN', 'LSTM', 'GRU'],
        'units': [32, 64, 128],
        'layers': [1, 2],
        'dropout': [0.1, 0.2, 0.3],
        'learning_rate': [0.001, 0.01],
        'batch_size': [16, 32, 64]
    }
    
    # Generate all combinations (sample a subset for demonstration)
    keys = list(param_grid.keys())
    combinations = list(itertools.product(*param_grid.values()))
    
    # Sample 20 combinations for demonstration
    np.random.seed(42)
    selected_combinations = np.random.choice(len(combinations), 
                                           min(20, len(combinations)), 
                                           replace=False)
    
    results = []
    
    print(f"Running hyperparameter search with {len(selected_combinations)} configurations...")
    
    for i, combo_idx in enumerate(selected_combinations):
        combo = combinations[combo_idx]
        params = dict(zip(keys, combo))
        
        print(f"\\nConfiguration {i+1}/{len(selected_combinations)}: {params}")
        
        try:
            # Create model with current parameters
            model = experiment.create_model(
                model_type=params['model_type'],
                units=params['units'],
                layers=params['layers'],
                dropout=params['dropout'],
                learning_rate=params['learning_rate'],
                input_shape=(X.shape[1], 1)
            )
            
            # Evaluate model
            result = experiment.evaluate_model(
                model, X_train, y_train_scaled, X_test, y_test_scaled,
                epochs=30, batch_size=params['batch_size'], verbose=0
            )
            
            # Store results
            result.update(params)
            results.append(result)
            
            print(f"RMSE: {result['rmse']:.4f}, R²: {result['r2']:.4f}")
            
        except Exception as e:
            print(f"Error with configuration: {e}")
            continue
    
    return results, X_train, y_train_scaled, X_test, y_test_scaled

# Run hyperparameter search
print("Starting comprehensive hyperparameter search...")
search_results, X_train_exp, y_train_exp, X_test_exp, y_test_exp = run_hyperparameter_search()

print(f"\\nCompleted hyperparameter search with {len(search_results)} successful configurations")
